In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import random
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

from FEX.models import fex
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
from generate_data import make_data, make_adjacency
adj_matrix = make_adjacency(100, 0.35, device='cpu')
data, derivatives = make_data(num_samples=4096, adjacency=adj_matrix)

In [ ]:
dim0 = fex.CoupledFEX('depth_2_tree_config', 'depth_2_tree_config', target_dim=0, poolsize=5, controller_epochs=250, num_fex_epochs=100, bfgs_epochs=30, controller_lr=0.003, finetune_lr=1e-3, finetune_epochs=5000)
dim0.fit(data, derivatives, adj_matrix, batch_size=64, num_workers=5)

In [ ]:
dim1 = fex.SingleFEX('depth_2_tree_config', target_dim=1, controller_epochs=200, num_fex_epochs=60, bfgs_epochs=20, num_finetune_epochs=5000, finetune_lr=1e-3).to(device)
dim1.fit(data, derivatives, num_workers=5)

In [ ]:
adj_matrix = make_adjacency(100, 0.35, device='cpu')
data, derivatives = make_data(num_samples=4096, adjacency=adj_matrix)
dim2 = fex.SingleFEX('depth_2_tree_config', target_dim=2, controller_epochs=215, num_fex_epochs=65, bfgs_epochs=20, num_finetune_epochs=10000, finetune_lr=2e-3).to(device)
dim2.fit(data, derivatives, num_workers=5)
print(dim2)

In [ ]:
epsilon = 0.15
a = 0.2
b = 0.2
c = -5.7

df = pd.read_csv('../HR/data/BA_Nnodes100_Adj_deg_10_0.csv', header=None)
adj_matrix = torch.as_tensor(
    df.values,
    device=device,
    dtype=torch.float64,
)
df = pd.read_csv('Rossler_timeseries.csv')
num_nodes = adj_matrix.shape[0]
data = df.values.reshape(len(df), num_nodes, -1)

data = torch.as_tensor(
    data,
    device=device,
    dtype=torch.float64,
)
num_nodes = data.shape[1]


omega = (1.0 + 0.1 * torch.randn(num_nodes, device=device, dtype=data.dtype))


def dim0_predict(
    state: torch.Tensor,
    adj_matrix: torch.Tensor,
    omega: torch.Tensor,
    epsilon: float = 0.15,
):
    x1 = state[:, 0]
    x2 = state[:, 1]
    x3 = state[:, 2]

    neighbor_sum = adj_matrix @ x1
    degree = adj_matrix.sum(dim=1)
    coupling = neighbor_sum - degree * x1

    dx1_dt = -omega * x2 - x3 + epsilon * coupling
    return dx1_dt.unsqueeze(1)

def dim1_predict(
    state: torch.Tensor,
    omega: torch.Tensor,
    a: float = 0.2,
):
    x1 = state[:, 0]
    x2 = state[:, 1]

    dx2_dt = omega * x1 + a * x2
    return dx2_dt.unsqueeze(1)


def dim2_predict(
    state: torch.Tensor,
    b: float = 0.2,
    c: float = -5.7,
):
    x1 = state[:, 0]
    x3 = state[:, 2]

    dx3_dt = b + x3 * (x1 + c)
    return dx3_dt.unsqueeze(1)

In [ ]:
def as_column(x: torch.Tensor) -> torch.Tensor:
    if x.ndim == 1:
        return x.unsqueeze(1).to(device)
    return x.to(device)


def rhs(state: torch.Tensor) -> torch.Tensor:
    return torch.cat(
        (
            as_column(dim0.predict(state, adj_matrix)),
            as_column(dim1.predict(state)),
            as_column(dim2.predict(state)),
        ),
        dim=1,
    )

def true_rhs(state: torch.Tensor):
    return torch.cat(
        (
            as_column(dim0_predict(state, adj_matrix, omega)),
            as_column(dim1_predict(state, omega, a=a)),
            as_column(dim2_predict(state, b=b, c=c)),
        ),
        dim=1,
    )

adj_matrix = torch.as_tensor(
    adj_matrix,
    device=device,
    dtype=data.dtype,
)

timesteps = len(data)
dt = 0.01

predicted_states = torch.empty(
    (timesteps + 1, *data.shape[1:]),
    device=device,
    dtype=data.dtype,
)

predicted_states[0].copy_(data[0])

with torch.inference_mode():
    for t in range(timesteps):
        state = predicted_states[t]

        k1 = rhs(state)
        k2 = rhs(state + 0.5 * dt * k1)
        k3 = rhs(state + 0.5 * dt * k2)
        k4 = rhs(state + dt * k3)

        next_state = state + (dt / 6.0) * (
            k1 + 2.0 * k2 + 2.0 * k3 + k4
        )

        predicted_states[t + 1].copy_(next_state)

        if not torch.isfinite(next_state).all():
            print(f"Non-finite state at timestep {t + 1}")
            break

ground_truth_states = torch.empty(
    (timesteps + 1, *data.shape[1:]),
    device=device,
    dtype=data.dtype,
) 
ground_truth_states[0].copy_(data[0])
with torch.inference_mode():
    for t in range(timesteps):
        state = ground_truth_states[t]

        k1 = true_rhs(state)
        k2 = true_rhs(state + 0.5 * dt * k1)
        k3 = true_rhs(state + 0.5 * dt * k2)
        k4 = true_rhs(state + dt * k3)

        next_state = state + (dt / 6.0) * (
            k1 + 2.0 * k2 + 2.0 * k3 + k4
        )

        ground_truth_states[t + 1].copy_(next_state)

        if not torch.isfinite(next_state).all():
            print(f"Non-finite state at timestep {t + 1}")
            break

In [ ]:
from FEX.utils.plots import plot_dynamics

node = 20
fig = plot_dynamics(data[:, node, 0].cpu(), data[:, node, 1].cpu(), data[:, node, 2].cpu(), predicted_states[:, node, :].cpu())
fig.show()